In [ ]:
from google.colab import drive
from pathlib import Path
import os
import zipfile

# 1. Montar Google Drive (solo pide autorización la primera vez por sesión)
drive.mount('/content/drive')

# 2. Rutas dentro de Drive — los datos persisten entre sesiones
BASE_DIR    = Path("/content/drive/MyDrive/binance_full_history")
EXTRACT_DIR = BASE_DIR / "files"

BASE_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Credenciales Kaggle
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

!pip install -q kaggle

DATASET = "jorijnsmit/binance-full-history"

# 4. Descargar solo si el ZIP no existe todavía en Drive
zip_files = list(BASE_DIR.glob("*.zip"))

if not zip_files:
    print("ZIP no encontrado en Drive — descargando (~26 GB)...")
    !kaggle datasets download -d {DATASET} -p {BASE_DIR}
    zip_files = list(BASE_DIR.glob("*.zip"))
else:
    print(f"ZIP ya existe en Drive ({zip_files[0].name}) — se omite la descarga.")

ZIP_PATH = zip_files[0]
print(f"ZIP: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / (1024**3):.2f} GB)")

# 5. Extraer solo si los parquet no existen todavía
parquet_files = sorted(EXTRACT_DIR.glob("*.parquet"))

if not parquet_files:
    print("Extrayendo archivos...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    parquet_files = sorted(EXTRACT_DIR.glob("*.parquet"))
    print("Extracción completa.")
else:
    print(f"{len(parquet_files)} archivos parquet ya extraídos en Drive — se omite la extracción.")

print(f"\nTotal parquet files: {len(parquet_files)}")
print("Primeros 20:")
for f in parquet_files[:20]:
    print(f.name)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when
import pyarrow.parquet as pq

spark = (
    SparkSession.builder
    .appName("Binance Full History Analysis")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.files.maxPartitionBytes", "128MB")
    .getOrCreate()
)

df = spark.read.parquet(str(EXTRACT_DIR / "*.parquet"))

df.printSchema()
print("Columnas:", len(df.columns))

# Read row counts from parquet footer metadata — no data scan, orders of magnitude faster
total_rows = sum(pq.read_metadata(str(f)).num_rows for f in parquet_files)
print(f"Registros (metadata): {total_rows:,}")

df.show(5, truncate=False)


# ESTADÍSTICAS GENERALES

In [ ]:
# Plain sample — no derived columns during scan so Spark keeps the vectorized Parquet reader
sample_df = df.sample(fraction=0.0001, seed=42).cache()
print(f"Registros en muestra: {sample_df.count():,}")

print("\n=== Estadísticas Generales (muestra) ===")
sample_df.describe().show(truncate=False)


# NULOS SOBRE MUESTRA

In [ ]:
print("=== Valores Nulos / Faltantes (muestra) ===")

null_counts = sample_df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dict(sample_df.dtypes)[c] in ("float", "double", "int", "bigint")
    else count(when(col(c).isNull(), c)).alias(c)
    for c in sample_df.columns
])

null_counts.show(truncate=False)


# RANGO TEMPORAL SOBRE DATASET

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

print("=== Rango Temporal del Dataset Completo ===")

# Parquet guarda estadísticas min/max por columna en los metadatos
# Spark las lee sin hacer full scan de los datos
df.select(
    spark_min("open_time").alias("Fecha más antigua"),
    spark_max("open_time").alias("Fecha más reciente")
).show(truncate=False)


=== Rango Temporal del Dataset Completo ===
+-------------------+-------------------+
|Fecha más antigua  |Fecha más reciente |
+-------------------+-------------------+
|2017-07-14 04:00:00|2022-11-17 22:22:00|
+-------------------+-------------------+



# PERCENTILES (DISTRIBUCIÓN)

In [ ]:
numeric_cols = ["open", "high", "low", "close", "volume", "quote_asset_volume", "number_of_trades"]
quantiles = [0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

print("=== Percentiles (p5 → p99) — muestra ===")
header = f"{'Columna':<35}" + "  ".join(f"p{int(q*100):>2}" for q in quantiles)
print(header)
print("-" * len(header))

for c in numeric_cols:
    vals = sample_df.approxQuantile(c, quantiles, relativeError=0.01)
    row = f"{c:<35}" + "  ".join(f"{v:>9.4g}" for v in vals)
    print(row)


# DISTRIBUCIÓN DE RETORNOS (LOG-RETURN)

In [ ]:
from pyspark.sql.functions import log, abs as spark_abs

returns_df = (
    sample_df
    .filter((col("open") > 0) & (col("close") > 0))
    .withColumn("log_return", log(col("close") / col("open")))
    .cache()
)

print("=== Estadísticas del Log-Return por Vela (close/open) — muestra ===")
returns_df.select(
    col("log_return"),
    spark_abs(col("log_return")).alias("abs_log_return"),
).describe().show(truncate=False)

print("\n=== Percentiles del Log-Return ===")
lr_quantiles = returns_df.approxQuantile("log_return", [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99], 0.001)
for label, val in zip(["p1", "p5", "p25", "p50", "p75", "p95", "p99"], lr_quantiles):
    print(f"  {label:>4}: {val:+.6f}")

print("\n=== Top 10 Velas con Mayor Retorno Positivo (muestra) ===")
returns_df.orderBy(col("log_return").desc()).select(
    "open_time", "open", "close", "log_return"
).show(10, truncate=False)

print("\n=== Top 10 Velas con Mayor Caída (muestra) ===")
returns_df.orderBy(col("log_return")).select(
    "open_time", "open", "close", "log_return"
).show(10, truncate=False)


# CORRELACIÓN ENTRE COLUMNAS NUMÉRICAS

In [ ]:
numeric_cols_corr = ["open", "close", "volume", "quote_asset_volume", "number_of_trades",
                     "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"]

# Single pandas conversion from the in-memory cache — far faster than 49 separate Spark corr() jobs
pdf = sample_df.select(numeric_cols_corr).toPandas()

print("=== Matriz de Correlación de Pearson (muestra) ===")
corr_matrix = pdf.corr()
print(corr_matrix.to_string(float_format=lambda x: f"{x:+.3f}"))


# ANÁLISIS POR PAR (DATASET COMPLETO)

`input_file_name()` desactiva el lector vectorizado de Parquet, por lo que nunca se añade a la muestra. En su lugar se hace **un único escaneo completo** aquí, se calculan todas las métricas por par en un solo `groupBy` y el resultado (~1 000 filas) se cachea. El resto de las secciones por par simplemente consultan ese dataframe pequeño.

In [ ]:
from pyspark.sql.functions import (
    input_file_name, regexp_extract, log, when,
    sum as spark_sum, avg, stddev,
    min as spark_min, max as spark_max,
    datediff, to_date,
)

df_with_pair = df.withColumn(
    "pair",
    regexp_extract(input_file_name(), r"([^/\\]+)\.parquet$", 1)
)

print("Computando estadísticas por par (escaneo único del dataset completo)...")
pair_stats = (
    df_with_pair
    .withColumn("is_zero_vol", when(col("volume") == 0, 1).otherwise(0))
    .withColumn(
        "log_return",
        when((col("open") > 0) & (col("close") > 0), log(col("close") / col("open")))
    )
    .groupBy("pair")
    .agg(
        spark_min("open_time").alias("inicio"),
        spark_max("open_time").alias("fin"),
        count("*").alias("num_candles"),
        spark_sum("is_zero_vol").alias("zero_vol"),
        spark_sum("quote_asset_volume").alias("total_quote_volume"),
        spark_sum("number_of_trades").alias("total_trades"),
        spark_min("close").alias("precio_min"),
        spark_max("close").alias("precio_max"),
        avg("close").alias("precio_medio"),
        stddev("log_return").alias("volatilidad"),
        avg("log_return").alias("retorno_medio"),
    )
    .withColumn("pct_zero", (col("zero_vol") / col("num_candles") * 100).cast("decimal(5,2)"))
    .withColumn("dias_activo", datediff(to_date(col("fin")), to_date(col("inicio"))))
    .cache()
)
n_pairs = pair_stats.count()
print(f"Listo: {n_pairs} pares cacheados.")


## TOP PARES POR VOLUMEN Y ACTIVIDAD

In [ ]:
print("=== Top 20 Pares por Volumen en Quote Asset ===")
(
    pair_stats
    .orderBy(col("total_quote_volume").desc())
    .select("pair", "total_quote_volume", "total_trades", "num_candles",
            "precio_min", "precio_max", "precio_medio")
    .show(20, truncate=False)
)


## COBERTURA TEMPORAL POR PAR

In [ ]:
print("=== Cobertura Temporal por Par — top 20 con más velas ===")
(
    pair_stats
    .orderBy(col("num_candles").desc())
    .select("pair", "inicio", "fin", "num_candles", "dias_activo")
    .show(20, truncate=False)
)


## VELAS CON VOLUMEN CERO

In [ ]:
totals = pair_stats.agg(
    spark_sum("num_candles").alias("total"),
    spark_sum("zero_vol").alias("zero_vol"),
).first()

print("=== Velas con Volumen Cero (dataset completo) ===")
print(f"Total velas  : {totals['total']:,}")
print(f"Volumen = 0  : {totals['zero_vol']:,}  ({totals['zero_vol'] / totals['total'] * 100:.2f}%)")
print(f"Volumen > 0  : {totals['total'] - totals['zero_vol']:,}  ({(totals['total'] - totals['zero_vol']) / totals['total'] * 100:.2f}%)")

print("\n=== Top 15 Pares con Mayor % de Velas Sin Volumen ===")
(
    pair_stats
    .orderBy(col("pct_zero").desc())
    .select("pair", "num_candles", "zero_vol", "pct_zero")
    .show(15, truncate=False)
)


# VOLATILIDAD POR PAR (DESVIACIÓN ESTÁNDAR DEL LOG-RETURN)

In [ ]:
print("=== Top 20 Pares Más Volátiles (dataset completo) ===")
(
    pair_stats
    .filter(col("num_candles") >= 100)
    .orderBy(col("volatilidad").desc())
    .select("pair", "volatilidad", "retorno_medio", "num_candles")
    .show(20, truncate=False)
)

print("\n=== Top 20 Pares Más Estables (dataset completo) ===")
(
    pair_stats
    .filter(col("num_candles") >= 100)
    .orderBy(col("volatilidad"))
    .select("pair", "volatilidad", "retorno_medio", "num_candles")
    .show(20, truncate=False)
)
